In [1]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.19.6
#   kernelspec:
#     display_name: Python (latest Jaspy)
#     language: python
#     name: jaspy
# ---

# NB 01 Unconditional copulas (per fold)

One unconditional copula of the pair (U1, U2) is fitted per blocked fold, on the fold's training steps, on the **kernel-conditional pseudo-observations** of notebook 00 (the marginals are conditional on the divergence; the copula itself is unconditional). Seven parametric families (Gaussian, t, Clayton, Gumbel, Gumbel (180°), Frank, Joe) and the unconditional CODINE net (3 seeds). Everything is repeated for the four folds; nothing is pooled across folds here (pooling happens on the per-step loss in notebook 05).

Per fold, the table reports training AIC, test CVPE of the fold, and the marginal-uniformity KS distance D_max of independently generated copula margins (a model is admissible only if D_max <= 0.07 in every required fold, seed and regime; no p-values or bootstrap replicates are used). Fits, gates and nets are computed here and reused by notebook 04 through shared resume files. Each table is written as LaTeX and CSV.

## Imports and per-fold inputs

In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "1"; os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"; os.environ["NUMEXPR_NUM_THREADS"] = "1"
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

from run_support import load_artifact, save_artifact, SMOKE, MAX_WORKERS   # production defaults; override with THESIS_SMOKE / THESIS_MAX_WORKERS (see run_support.py)
from copula_families import signed_params_of, branch_of, fit_and_evaluate, DEFAULT_FAMILIES   # primary parameter WITH the sign of the branch (negative = turned rotation)
# Unconditional models / penalty calibration do not use previously calibrated network weights.
os.environ["THESIS_ALLOW_UNCALIBRATED_IMPORT"] = "1"
import fold_pipeline as FP
import reporting_tables
reporting_tables.configure("01_unconditional_copulas")

FOLDS = FP.FOLDS
ARR = FP.load_all_folds()          # per-fold arrays: kernel-conditional u1/u2, masks, b, w
print("kernel bandwidth h by fold =", load_artifact("preprocessing_folds_summary").get('kernel_h_by_fold'))

# Fits, marginal KS checks and losses of the seven parametric families (the same jobs notebook 04 uses: whichever
# notebook runs first computes them, the other resumes from the shared files in artifacts/fold_batch).
PARAM, _ = FP.run_param_stage(ARR, max_workers=MAX_WORKERS)
# Unconditional CODINE: 4 folds x 3 seeds; marginal KS check on seed 0 only.
NEURAL = FP.run_batch(FP.build_fold_jobs(ARR, None, smoke=SMOKE, kinds=("codine_uncond",)),
                      max_workers=MAX_WORKERS, batch_dir=FP.BATCH_DIR, desc='unconditional CODINE')
NLOSS = FP.neural_results_to_losses(NEURAL)[0]      # seed-averaged per-fold loss, same rule as notebooks 04 and 05

kernel bandwidth h by fold = {'A1': 0.22832144093183465, 'A2': 0.6636094179528018, 'B1': 0.6643855353375984, 'B2': 0.6643855353375984}
parametric folds: 0 valid resume results, 8 assigned jobs to run


parametric folds: 8 workers (CPU/task limit 8, estimated 1.5 GiB each; RAM headroom checked at pool creation)


parametric folds:   0%|          | 0/8 [00:00<?, ?it/s]

unconditional CODINE: 0 valid resume results, 12 assigned jobs to run


unconditional CODINE: 12 workers (CPU/task limit 12, estimated 1.5 GiB each; RAM headroom checked at pool creation)


unconditional CODINE:   0%|          | 0/12 [00:00<?, ?it/s]

## Parametric families per fold: AIC, test CVPE, marginal KS check

In [3]:
import gate_config as GC

RES = {}
TABLES = {}
for f in FOLDS:
    A_ = ARR[f]; tr, te = A_['train'], A_['test']
    _, rows = fit_and_evaluate(A_['u1'], A_['u2'], train_mask=tr, test_mask=te, families=list(DEFAULT_FAMILIES), quiet=True)
    RES[f] = rows
    tab = pd.DataFrame([dict(Family=r['name'], Branch=branch_of(r['cop']), AIC_train=r['aic'], CVPE_test=float(np.nanmean(PARAM[f]['loss'][f"Uncond:{r['name']}"])), KS_D=PARAM[f].get('ks', {}).get(('uncond', r['name']), np.nan)) for r in rows])
    tab['Admissible'] = [GC.ks_seed_pass(d) for d in tab.KS_D]
    tab.loc[len(tab)] = dict(Family='CODINE', Branch='-', AIC_train=np.nan, CVPE_test=float(np.nanmean(NLOSS[FP.UNCOND_CODINE][f])), KS_D=max((NEURAL.get(FP.net_key(f, seed, FP.UNCOND_CODINE), {}).get('ks', np.nan) for seed in FP.SEEDS), default=np.nan), Admissible=all(GC.ks_seed_pass(NEURAL.get(FP.net_key(f, seed, FP.UNCOND_CODINE), {}).get('ks')) for seed in FP.SEEDS))
    TABLES[f] = tab
    print(f"\n=== fold {f}: train n={int(tr.sum())}, test n={int(te.sum())}, b={A_['b']} ===")
    display(tab)


=== fold A1: train n=1394, test n=1393, b=49 ===


,Family,Branch,AIC_train,CVPE_test,KS_D,Admissible
0,Gaussian,-,-26.974899,0.151733,0.025275,True
1,t,-,-25.075838,0.151708,0.024607,True
2,Clayton,own,-16.105711,0.151366,0.022888,True
3,Gumbel,own,-27.005665,0.151097,0.025041,True
4,Gumbel (180°),own,-14.115379,0.151250,0.022012,True
5,Frank,-,-23.736520,0.151538,0.024776,True
6,Joe,own,-21.147063,0.150562,0.027291,True
7,CODINE,-,NaN,0.151846,0.042198,True



=== fold A2: train n=1393, test n=1394, b=65 ===


,Family,Branch,AIC_train,CVPE_test,KS_D,Admissible
0,Gaussian,-,-0.228190,0.182277,0.023025,True
1,t,-,-16.298552,0.182137,0.023689,True
2,Clayton,own,0.578238,0.182869,0.023888,True
3,Gumbel,own,-8.322650,0.181486,0.021766,True
4,Gumbel (180°),own,-2.468606,0.182174,0.023625,True
5,Frank,-,-0.756565,0.182219,0.024197,True
6,Joe,own,-9.439397,0.181898,0.023231,True
7,CODINE,-,NaN,0.182693,0.037603,True



=== fold B1: train n=1345, test n=1345, b=38 ===


,Family,Branch,AIC_train,CVPE_test,KS_D,Admissible
0,Gaussian,-,-13.597034,0.159732,0.022609,True
1,t,-,-15.283646,0.159730,0.024285,True
2,Clayton,own,-9.274856,0.160357,0.022943,True
3,Gumbel,own,-17.038860,0.159700,0.024749,True
4,Gumbel (180°),own,-7.628889,0.160172,0.024002,True
5,Frank,-,-11.688191,0.159736,0.022614,True
6,Joe,own,-13.815923,0.159936,0.026714,True
7,CODINE,-,NaN,0.160142,0.051843,True



=== fold B2: train n=1345, test n=1345, b=55 ===


,Family,Branch,AIC_train,CVPE_test,KS_D,Admissible
0,Gaussian,-,-12.255750,0.168986,0.022786,True
1,t,-,-19.565354,0.168988,0.024849,True
2,Clayton,own,-8.302582,0.169336,0.023757,True
3,Gumbel,own,-16.645573,0.169035,0.025342,True
4,Gumbel (180°),own,-12.132369,0.169191,0.023666,True
5,Frank,-,-11.913730,0.169122,0.022646,True
6,Joe,own,-13.908957,0.169231,0.027840,True
7,CODINE,-,NaN,0.169036,0.047441,True


## Write the per-fold tables

In [4]:
for f in FOLDS:
    reporting_tables.write_table(TABLES[f].set_index('Family'), f'unconditional_copulas_fold_{f}',
        f'Unconditional copulas, fold {f}: training AIC, test CVPE and marginal KS distance.',
        label=f'tab:uncond_fold_{f}', col_format='llrrrc')

## Save

In [5]:
save_artifact("unconditional_copulas_folds",
              tables=TABLES, folds=list(FOLDS),
              family_params={f: {r['name']: signed_params_of(r['cop']) for r in RES[f]} for f in FOLDS},
              family_names_to_pvfamily={f: {r['name']: r['fam'] for r in RES[f]} for f in FOLDS})
print("saved unconditional_copulas_folds")

saved unconditional_copulas_folds


## Table layout

Every table with a tabular is wrapped so that it shrinks to the text width when it is wider and is never enlarged (`reporting_tables.fit_all_tables`); no table needs a manual size edit in the report.

In [6]:
import reporting_tables
print('table files fitted to the text width:', reporting_tables.fit_all_tables())

table files fitted to the text width: []
